# Qwen2.5-VL Text-Generation Attack with EOT-PGD

This notebook copies `attack_qwen25vl_text_generation.ipynb` into an interactive workflow for a targeted Expectation over Transformation (EOT)-PGD attack against Qwen2.5-VL. It perturbs a source image so the model is pushed toward generating a fixed target caption for a fixed user prompt while optimizing across random camera-like transforms.

The EOT transform stack simulates a general camera capture path using:
- random perspective
- random crop and resize back to the original canvas
- random rotation with white fill
- color jitter

Runtime assumptions:
- Launch the notebook from this repository or one of its subdirectories.
- A CUDA device is available.
- The Qwen2.5-VL weights are accessible to `transformers`.

Expected outputs:
- `results/qwen25vl_textgen_eot_adv.png`
- `results/qwen25vl_textgen_eot_noise.png`
- `results/qwen25vl_textgen_eot_generations.txt`


In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repo root from the current working directory. "
        "Launch the notebook from this repository or one of its subdirectories."
    )


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"

print(f"Repo root: {REPO_ROOT}")
print(f"Results dir: {RESULTS_DIR}")


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
SOURCE_IMAGE_PATH = REPO_ROOT / "data" / "images" / "worksheet_000002.png"
USER_PROMPT = "Solve the math problems shown in the image. Show steps and give the final answers."
TARGET_TEXT = "The image shows a toaster sitting on a clean kitchen counter."

EPSILON = 64 / 255
ALPHA = 1 / 255
STEPS = 1000
MODEL_INPUT_SIZE = 448
MAX_NEW_TOKENS = 128

EOT_SAMPLES = 5
EVAL_EOT_SAMPLES = 8
ROTATION_DEGREES = 10
PERSPECTIVE_DISTORTION = 0.4
CROP_SCALE = (0.8, 1.0)
CROP_RATIO = (0.9, 1.1)
COLOR_JITTER_BRIGHTNESS = 0.1
COLOR_JITTER_CONTRAST = 0.1
COLOR_JITTER_SATURATION = 0.1

OUTPUT_ADV_PATH = RESULTS_DIR / "qwen25vl_textgen_eot_adv.png"
OUTPUT_NOISE_PATH = RESULTS_DIR / "qwen25vl_textgen_eot_noise.png"
OUTPUT_REPORT_PATH = RESULTS_DIR / "qwen25vl_textgen_eot_generations.txt"

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

INSPECTION_PROMPT = "What is shown in this image? Describe it in detail."

print(f"Source image: {SOURCE_IMAGE_PATH}")
print(f"Prompt: {USER_PROMPT}")
print(f"Target text: {TARGET_TEXT}")
print(f"EOT samples per step: {EOT_SAMPLES}")
print(f"EOT evaluation samples: {EVAL_EOT_SAMPLES}")


In [ ]:
def load_image_tensor(image_path: str | Path, device: torch.device) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    return transforms.ToTensor()(image).to(device).unsqueeze(0)


def sample_camera_transform(image_tensor: torch.Tensor) -> torch.Tensor:
    if image_tensor.ndim != 3:
        raise ValueError("Expected image_tensor with shape (C, H, W).")

    channels, height, width = image_tensor.shape
    fill = [1.0] * channels

    startpoints, endpoints = transforms.RandomPerspective.get_params(
        width=width,
        height=height,
        distortion_scale=PERSPECTIVE_DISTORTION,
    )
    x = TF.perspective(
        image_tensor,
        startpoints=startpoints,
        endpoints=endpoints,
        interpolation=InterpolationMode.BILINEAR,
        fill=fill,
    )

    top, left, crop_height, crop_width = transforms.RandomResizedCrop.get_params(
        x,
        scale=list(CROP_SCALE),
        ratio=list(CROP_RATIO),
    )
    x = TF.resized_crop(
        x,
        top,
        left,
        crop_height,
        crop_width,
        size=[height, width],
        interpolation=InterpolationMode.BILINEAR,
        antialias=True,
    )

    angle = float(torch.empty(1).uniform_(-ROTATION_DEGREES, ROTATION_DEGREES).item())
    x = TF.rotate(
        x,
        angle=angle,
        interpolation=InterpolationMode.BILINEAR,
        fill=fill,
    )

    brightness_factor = 1.0 + float(
        torch.empty(1).uniform_(-COLOR_JITTER_BRIGHTNESS, COLOR_JITTER_BRIGHTNESS).item()
    )
    contrast_factor = 1.0 + float(
        torch.empty(1).uniform_(-COLOR_JITTER_CONTRAST, COLOR_JITTER_CONTRAST).item()
    )
    saturation_factor = 1.0 + float(
        torch.empty(1).uniform_(-COLOR_JITTER_SATURATION, COLOR_JITTER_SATURATION).item()
    )

    x = TF.adjust_brightness(x, brightness_factor)
    x = TF.adjust_contrast(x, contrast_factor)
    x = TF.adjust_saturation(x, saturation_factor)
    return torch.clamp(x, 0.0, 1.0)


def pack_for_qwen(
    image_tensor: torch.Tensor,
    *,
    model_input_size: int,
    mean: torch.Tensor,
    std: torch.Tensor,
    patch_size: int,
    temporal_patch_size: int,
    merge_size: int,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    height, width = image_tensor.shape[-2:]
    scale = min(model_input_size / height, model_input_size / width)
    resized_h = max(1, int(round(height * scale)))
    resized_w = max(1, int(round(width * scale)))

    x = F.interpolate(
        image_tensor.unsqueeze(0),
        size=(resized_h, resized_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)

    pad_h = model_input_size - resized_h
    pad_w = model_input_size - resized_w
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    x = F.pad(x, (pad_left, pad_right, pad_top, pad_bottom), value=1.0)
    x = (x - mean) / std

    frames = x.unsqueeze(0)
    if frames.shape[0] % temporal_patch_size != 0:
        repeats = temporal_patch_size - (frames.shape[0] % temporal_patch_size)
        frames = torch.cat([frames, frames[-1:].repeat(repeats, 1, 1, 1)], dim=0)

    channels = frames.shape[1]
    grid_t = frames.shape[0] // temporal_patch_size
    grid_h = frames.shape[2] // patch_size
    grid_w = frames.shape[3] // patch_size

    patches = frames.reshape(
        grid_t,
        temporal_patch_size,
        channels,
        grid_h // merge_size,
        merge_size,
        patch_size,
        grid_w // merge_size,
        merge_size,
        patch_size,
    )
    patches = patches.permute(0, 3, 6, 4, 7, 2, 1, 5, 8)

    pixel_values = patches.reshape(
        grid_t * grid_h * grid_w,
        channels * temporal_patch_size * patch_size * patch_size,
    )
    image_grid_thw = torch.tensor([[grid_t, grid_h, grid_w]], device=device, dtype=torch.long)
    return pixel_values, image_grid_thw


def save_noise_visualization(delta: torch.Tensor, output_path: Path) -> None:
    noise = torch.clamp(delta.squeeze(0).cpu() * 10 + 0.5, 0.0, 1.0)
    transforms.ToPILImage()(noise).save(output_path)


def show_saved_results(image_paths: tuple[Path, ...], report_path: Path) -> None:
    for path in image_paths:
        print(path)
        if path.exists():
            display(Image.open(path))
        else:
            print("Missing output file.")

    print(report_path)
    if report_path.exists():
        print(report_path.read_text())
    else:
        print("Missing report file.")


def format_generation_block(title: str, generations: list[str]) -> list[str]:
    lines = [title]
    for index, generation in enumerate(generations, start=1):
        lines.append(f"[{index}] {generation}")
    return lines


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA device is required for this notebook.")

device = torch.device("cuda:1")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"[Info] Loading model: {MODEL_NAME}")
processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=False)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)
model.eval()
model.requires_grad_(False)

x_clean = load_image_tensor(SOURCE_IMAGE_PATH, device)

vision_config = model.config.vision_config
patch_size = vision_config.patch_size
temporal_patch_size = vision_config.temporal_patch_size
merge_size = vision_config.spatial_merge_size
mean = torch.tensor(CLIP_MEAN, device=device).view(3, 1, 1)
std = torch.tensor(CLIP_STD, device=device).view(3, 1, 1)

print(f"Loaded source image tensor with shape: {tuple(x_clean.shape)}")


In [ ]:
def build_prompt_inputs(prompt: str) -> tuple[str, torch.Tensor, torch.Tensor, torch.Tensor]:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    rendered_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    dummy_image = Image.new("RGB", (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), color="white")
    prompt_inputs = processor(text=[rendered_prompt], images=[dummy_image], return_tensors="pt")

    input_ids = prompt_inputs["input_ids"].to(device)
    attention_mask = prompt_inputs["attention_mask"].to(device)
    mm_token_type_ids = prompt_inputs.get("mm_token_type_ids")
    if mm_token_type_ids is None:
        mm_token_type_ids = torch.zeros_like(input_ids, dtype=torch.int32)
    mm_token_type_ids = mm_token_type_ids.to(device)
    return rendered_prompt, input_ids, attention_mask, mm_token_type_ids


prompt_text, prompt_input_ids, prompt_attention_mask, prompt_mm_token_type_ids = build_prompt_inputs(USER_PROMPT)

tokenizer = processor.tokenizer
target_ids = tokenizer(TARGET_TEXT, add_special_tokens=False, return_tensors="pt")["input_ids"].to(device)
eos_token_id = tokenizer.eos_token_id
if eos_token_id is not None and (target_ids.shape[1] == 0 or target_ids[0, -1].item() != eos_token_id):
    target_ids = torch.cat([target_ids, torch.tensor([[eos_token_id]], device=device)], dim=1)

full_input_ids = torch.cat([prompt_input_ids, target_ids], dim=1)
full_attention_mask = torch.cat([prompt_attention_mask, torch.ones_like(target_ids)], dim=1)
full_mm_token_type_ids = torch.cat(
    [
        prompt_mm_token_type_ids,
        torch.zeros(target_ids.shape, device=device, dtype=prompt_mm_token_type_ids.dtype),
    ],
    dim=1,
)
labels = full_input_ids.clone()
labels[:, : prompt_input_ids.shape[1]] = -100


def target_loss(pixel_values: torch.Tensor, image_grid_thw: torch.Tensor) -> torch.Tensor:
    outputs = model(
        input_ids=full_input_ids,
        attention_mask=full_attention_mask,
        mm_token_type_ids=full_mm_token_type_ids,
        pixel_values=pixel_values,
        image_grid_thw=image_grid_thw,
        labels=labels,
        return_dict=True,
    )
    return outputs.loss


def pack_image(image_tensor: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return pack_for_qwen(
        image_tensor,
        model_input_size=MODEL_INPUT_SIZE,
        mean=mean,
        std=std,
        patch_size=patch_size,
        temporal_patch_size=temporal_patch_size,
        merge_size=merge_size,
        device=device,
    )


def compute_target_loss_from_image(image_tensor: torch.Tensor) -> float:
    pixel_values, image_grid_thw = pack_image(image_tensor)
    return float(target_loss(pixel_values, image_grid_thw).item())


def decode_generation(generated: torch.Tensor, input_ids: torch.Tensor) -> str:
    new_tokens = generated[:, input_ids.shape[1] :]
    return processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


def generate_from_image_with_prompt(
    image_tensor: torch.Tensor,
    *,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    mm_token_type_ids: torch.Tensor,
) -> str:
    pixel_values, image_grid_thw = pack_image(image_tensor)
    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        mm_token_type_ids=mm_token_type_ids,
        pixel_values=pixel_values,
        image_grid_thw=image_grid_thw,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )
    return decode_generation(generated, input_ids)


def generate_from_image(image_tensor: torch.Tensor) -> str:
    return generate_from_image_with_prompt(
        image_tensor,
        input_ids=prompt_input_ids,
        attention_mask=prompt_attention_mask,
        mm_token_type_ids=prompt_mm_token_type_ids,
    )


def evaluate_eot(
    image_tensor: torch.Tensor,
    *,
    num_samples: int,
    return_images: bool = False,
) -> tuple[float, list[str], list[torch.Tensor]]:
    losses: list[float] = []
    generations: list[str] = []
    sampled_images: list[torch.Tensor] = []

    for _ in range(num_samples):
        transformed_image = sample_camera_transform(image_tensor)
        pixel_values, image_grid_thw = pack_image(transformed_image)
        losses.append(float(target_loss(pixel_values, image_grid_thw).item()))
        generations.append(generate_from_image(transformed_image))
        if return_images:
            sampled_images.append(transformed_image.detach().cpu())

    mean_loss = sum(losses) / len(losses)
    return mean_loss, generations, sampled_images


print(prompt_text)


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with torch.no_grad():
    clean_loss = compute_target_loss_from_image(x_clean.squeeze(0))

delta = torch.zeros_like(x_clean, requires_grad=True)

print("[Info] Starting EOT-PGD text-generation attack...")
progress = tqdm(range(STEPS))
for _ in progress:
    delta.grad = None

    x_adv = torch.clamp(x_clean + delta, 0.0, 1.0)
    loss = 0.0
    for _ in range(EOT_SAMPLES):
        transformed_adv = sample_camera_transform(x_adv.squeeze(0))
        pixel_values, image_grid_thw = pack_image(transformed_adv)
        loss = loss + target_loss(pixel_values, image_grid_thw)

    loss = loss / EOT_SAMPLES
    loss.backward()

    with torch.no_grad():
        delta -= ALPHA * delta.grad.sign()
        delta.clamp_(-EPSILON, EPSILON)
        delta.copy_(torch.clamp(x_clean + delta, 0.0, 1.0) - x_clean)

    progress.set_postfix(loss=f"{loss.item():.4f}")

x_final = torch.clamp(x_clean + delta, 0.0, 1.0)

with torch.no_grad():
    adv_loss = compute_target_loss_from_image(x_final.squeeze(0))
    clean_output = generate_from_image(x_clean.squeeze(0))
    adv_output = generate_from_image(x_final.squeeze(0))
    clean_eot_loss, clean_eot_generations, _ = evaluate_eot(
        x_clean.squeeze(0),
        num_samples=EVAL_EOT_SAMPLES,
        return_images=False,
    )
    adv_eot_loss, adv_eot_generations, transformed_adv_samples = evaluate_eot(
        x_final.squeeze(0),
        num_samples=EVAL_EOT_SAMPLES,
        return_images=True,
    )

transforms.ToPILImage()(x_final.squeeze(0).cpu()).save(OUTPUT_ADV_PATH)
save_noise_visualization(delta, OUTPUT_NOISE_PATH)

report_lines = [
    f"Prompt: {prompt_text}",
    f"Target text: {TARGET_TEXT}",
    f"Deterministic clean target loss: {clean_loss:.6f}",
    f"Deterministic adversarial target loss: {adv_loss:.6f}",
    f"EOT mean clean target loss ({EVAL_EOT_SAMPLES} samples): {clean_eot_loss:.6f}",
    f"EOT mean adversarial target loss ({EVAL_EOT_SAMPLES} samples): {adv_eot_loss:.6f}",
    "",
    "Deterministic clean generation:",
    clean_output,
    "",
    "Deterministic adversarial generation:",
    adv_output,
    "",
    *format_generation_block("EOT clean generations:", clean_eot_generations),
    "",
    *format_generation_block("EOT adversarial generations:", adv_eot_generations),
]
OUTPUT_REPORT_PATH.write_text("\n".join(report_lines))

print(f"[Info] Deterministic clean target loss: {clean_loss:.6f}")
print(f"[Info] Deterministic adversarial target loss: {adv_loss:.6f}")
print(f"[Info] EOT mean clean target loss: {clean_eot_loss:.6f}")
print(f"[Info] EOT mean adversarial target loss: {adv_eot_loss:.6f}")
print("\n===== DETERMINISTIC CLEAN GENERATION =====\n")
print(clean_output)
print("\n===== DETERMINISTIC ADVERSARIAL GENERATION =====\n")
print(adv_output)
print(f"\n[Success] Saved adversarial image to {OUTPUT_ADV_PATH}")
print(f"[Success] Saved perturbation visualization to {OUTPUT_NOISE_PATH}")
print(f"[Success] Saved text report to {OUTPUT_REPORT_PATH}")


In [ ]:
show_saved_results((OUTPUT_ADV_PATH, OUTPUT_NOISE_PATH), OUTPUT_REPORT_PATH)


# Robust Inspection

Sample a few transformed adversarial views and run an inspection prompt against each one.


In [ ]:
inspection_prompt_text, inspection_input_ids, inspection_attention_mask, inspection_mm_token_type_ids = build_prompt_inputs(
    INSPECTION_PROMPT
)

print(inspection_prompt_text)

num_inspection_samples = min(4, len(transformed_adv_samples))
for index in range(num_inspection_samples):
    transformed_image_cpu = transformed_adv_samples[index]
    transformed_image = transformed_image_cpu.to(device)
    inspection_output = generate_from_image_with_prompt(
        transformed_image,
        input_ids=inspection_input_ids,
        attention_mask=inspection_attention_mask,
        mm_token_type_ids=inspection_mm_token_type_ids,
    )

    print(f"\n===== TRANSFORMED ADV SAMPLE {index + 1} =====\n")
    display(transforms.ToPILImage()(transformed_image_cpu))
    print(inspection_output)
